# Setup

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# read files from google drive
import os
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/Stanford_DeepRL_Spring2026/hw2'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)

# import modules from google drive
import sys
sys.path.append(GOOGLE_DRIVE_PATH)

Mounted at /content/drive


In [ ]:
%cd $GOOGLE_DRIVE_PATH

/content/drive/My Drive/Colab Notebooks/Stanford_DeepRL_Spring2026/hw2


In [ ]:
!pip install --quiet modal wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.4/979.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 22.7 MB/s eta 0:00:00


In [ ]:
!modal token set \
    --token-id YOUR_TOKEN_ID \
    --token-secret YOUR_TOKEN_SECRET

In [ ]:
os.environ['MODAL_LOGLEVEL'] = 'ERROR'

# Problem 1

We implement a Q-learning agent on a 2D grid world with discrete states and discrete actions. The grid world has two goals at different distances from the start. The grid is 5 × 4 (i.e. there are 20 states), with the agent always starting at $(0, 0)$, **Goal 2** at $(4, 0)$ (4 steps away), and **Goal 1** at $(4, 3)$ (7 steps away). At each step the agent chooses from four actions: $\{left, right, up, down\}$, which move it one step on the grid. If the agent attempts to move off the edge, the agent stays in place.

## Algorithm: Tabular Q-Learning
Instead of using a neural network to represent the Q-function, the agent maintains a table of Q-values $Q(s, a)$ initialized to zero, with one Q-value for each state and action (i.e. the table contains 20 × 4 = 80 scalar Q-values). When making updates, instead of updating the parameters of a neural network, we update the Q-values in the table directly following this Q-learning update:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$
where $\alpha$ is the learning rate and $\gamma$ is the discount factor.

## Your Tasks
1. **Implement `choose_action` in `grid_world_q_learning.py`.**  
Complete the `choose_action` function. With probability $\epsilon$ the agent selects a uniformly random action; otherwise it selects the greedy action $a^{\ast} = arg\max_a Q(s, a)$. Break ties arbitrarily.

2. **Implement `train_q_learning` in `grid_world_q_learning.py`.**  
Complete the inner loop of `train_q_learning`. For each step in an episode:  
(a) Choose an action using $\epsilon$-greedy policy.  
(b) Step the environment to obtain $(s_{t+1}, r_t, done)$.  
(c) Apply the Q-learning update above.  
(d) Terminate the episode if `done` is `True`.  

3. **Three Scenarios.**  
Run `python gridworld_q_learning.py` on each scenario below and report the learned trajectory and total reward.  
    * **Scenario 1** $(r_{step} = -1, R_1 = 10, R_2 = 5)$
    * **Scenario 2** $(r_{step} = -2, R_1 = 10, R_2 = 5)$
    * **Scenario 3** $(r_{step} = +1, R_1 = 1, R_2 = 1)$

    <font color="red">**For each scenario, report whether it reaches a goal, and if so which one it reaches. Give a one sentence explanation for why it learned that trajectory in each scenario.**</font>

In [ ]:
# 2>&1 | grep -v "modal.com": filter out links and progress animation clutter
!modal run modal_gridworld_q_learning.py 2>&1 | grep -v "modal.com"

✓ Initialized. View run at 
✓ Created objects.
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/modal_gridworld_q_learning.py
├── 🔨 Created mount PythonPackage:gridworld_q_learning
└── 🔨 Created function train.
/root/gridworld_q_learning.py
/root/gridworld_q_learning.py
scenario_1
Stopping app - local entrypoint completed.
  expected: goal_1
  observed: goal_1
  trajectory: ((0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3))
  actions: ('right', 'right', 'right', 'up', 'right', 'up', 'up')
  total_reward: 3.00
  start_q_values: (1.219, 2.265, 1.219, 2.265)
scenario_2
  expected: goal_2
  observed: goal_2
  trajectory: ((0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
  actions: ('right', 'right', 'right', 'right')
  total_reward: -3.00
  start_q_values: (-4.996, -3.057, -4.996, -4.329)
scenario_3
  expected: timeout
  observed: timeout
  trajectory: ((0, 0), (0, 0), (0, 1), (0, 2), (0, 2), (1, 2), (2, 2), (2, 3), (2, 2), (1, 2), (1, 3),

* **Scenario 1** $(r_{step} = -1, R_1 = 10, R_2 = 5)$  
Yes. The agent learns to reach Goal 1 because its higher terminal bonus ($R_1 = 10$) outweighs the larger step penalties incurred by the longer path, resulting in a higher total reward ($3.0$) than Goal 2 ($1.0$).

* **Scenario 2** $(r_{step} = -2, R_1 = 10, R_2 = 5)$  
Yes. As the step penalty becomes $-2$, the cost of the extra steps required to reach the distant Goal 1 dominates, making the shorter path to Goal 2 the optimal policy with a higher total reward ($-3.0$ vs. $-4.0$).

* **Scenario 3** $(r_{step} = +1, R_1 = 1, R_2 = 1)$  
No. Since the step reward is positive ($+1$) and the terminal bonuses are low ($R_1 = R_2 = 1$), the agent learns to avoid early termination by wandering around or bumping into walls to maximize its accumulated step rewards across the full horizon.

# Problem 2

We will implement an on-policy agent that learns to solve the manipulation task of hammering a nail, which collects rollouts under the current policy and then performs multiple epochs of gradient updates.

The algorithm you will implement is PPO, which uses a clipped surrogate objective. The full agent is implemented in `on_policy.py` and consists of:
* An **actor** $\pi_\theta(a_t|s_t)$ that outputs a `TruncatedNormal` distribution over actions.
* A **value function** $V_\phi(s_t)$ that estimates the state value function.
* A frozen **reference actor** $\pi_{\theta_{ref}}$ , a snapshot of the policy taken after behavior-cloning pre-training.

You should familiarize yourself with the `PPOAgent` class and its helper methods before starting. You should not modify any files other than `on_policy.py`.

## Your Tasks
1. **Generalized Advantage Estimation (GAE).**  
Complete the `compute_gae` method of the `PPOAgent` class.

2. **PPO Update.**  
Complete the two ### YOUR CODE HERE ### blocks inside the `update` method.
    * (a) Compute GAE advantages and returns (before the epoch loop).
    * (b) Clipped surrogate objective (inside the minibatch loop).

3. **Training.**  
Once your implementation is complete, train PPO with the command:
    ```
    modal run modal_on_policy.py
    ```
    <font color="red">**Attach a screenshot of the plot `eval/episode_success` for this run up to 1 million steps from Wandb. You should achieve at least 25% success rate.**</font>




In [ ]:
# executing for about 1.7 hr
!modal run modal_on_policy.py 2>&1 | grep -v "modal.com"

✓ Initialized. View run at 
✓ Created objects.
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/modal_on_policy.py
├── 🔨 Created mount PythonPackage:on_policy, PythonPackage:train_on_policy, 
│   PythonPackage:mw, PythonPackage:utils, PythonPackage:logger, 
│   PythonPackage:replay_buffer, PythonPackage:video
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/cfgs
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/demos
└── 🔨 Created function train.
/root/on_policy.py
/root/cfgs/on_policy_config.yaml
/root/on_policy.py
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymn

# Problem 3

We will implement a general actor-critic algorithm that learns to solve the task with almost 100% success rate. At each step the agent processes states $s_t$, which consists of the environment states (e.g., pose of the robot gripper and hammer, etc.) from the last two environment steps. The full algorithm consists of the following components, each in `off_policy.py`.
* The actor-critic algorithm utilizes critic networks $Q_{\phi^i} (s_t, a_t)$. This is implemented as the `Critic` class in `off_policy.py`. We use an ensemble with $i = 1 . . . N$ different critic networks. Using an ensemble of critic networks can help reduce error in the Q-value
estimates.
* Each critic network has a corresponding *target* critic network $\bar{Q}_{\phi^i} (s_t, a_t)$, which is used for computing the Bellman targets for critic updates. We use target critic networks, which are updated more slowly, to have a more stable target for learning.
* The final component of the algorithm is the Actor or policy $\pi_\theta(a_t|s_t)$. This is implemented as the Actor class in `off_policy.py`.

You should familiarize yourself with these functions and their inputs and outputs. You should not modify any other files besides `off_policy.py`.

## Your Tasks
1. **BC Pretraining.**  
In `off_policy.py` complete the `bc` function of the `ACAgent` class to train the policy using supervised behavior cloning.

2. **Update Agent.**  
Your implementation will be in the `ACAgent` class.
    * We begin by implementing the `update_critic` method using the Bellman objective. Consider transitions $(s_t, a_t, r_t, s_{t+1}, γ)$ and implement the following steps:  
    (a) Sample next state actions from the policy $a_{t+1} ∼ \pi_\theta(s_{t+1})$.  
    (b) Compute the Bellman targets.  
    (c) Compute the loss.  
    (d) Take a gradient step with respect to the critic parameters.  
    (e) Update the target critic parameters using an exponential moving average.

    * Next, we will improve the policy in the `update_actor` method.

    * **Training.**  
    Once you are done, run the RL fine-tuning with
        ```
        modal run modal_off_policy.py
        ```
        <font color="red">**Attach a screenshot of the plot `eval/episode_success` for this run from Wandb up to 100k steps. You should achieve a success rate of at least 90% before 100K environment steps.**</font>

    * **Analysis of UTD choice.**  
    In the final part, we will explore some optimization parameter choices. The update-to-data (UTD) ratio refers to the number of critic gradient steps we do after each environment step taken. So far we have used only 2 critics and UTD of 1. Repeat the previous part but change the argv inside the subprocess of `modal_off_policy.py` to use 10 `num_critics` and `utd = 5` as follows:
        ```
        result = subprocess.run(
            [
                "python", "-u", "train_off_policy.py",
                "--config-path", "/root/cfgs",
                "--config-name", "off_policy_config",
                "agent.num_critics=10",
                "utd=5"
            ],
        ```
        And then run again, but now with the new parameters:
        ```
        modal run modal_off_policy.py
        ```
        Here, we perform critic gradient steps 5x more often.  
        <font color="red">**Attach a screenshot of the plot `eval/episode_success` for this run up to 40k steps and compare it to the previous question. You should achieve a success rate of at least 90% before 40k environment steps. Provide a brief, one sentence explanation of why we observe these effects.**</font>  
        **Note: this is slower to run due to the higher computational cost and you should expect run time of about 2 hours for 50,000 steps.**


In [ ]:
# executing for about 0.7 hr
# agent.num_critics=2 utd=1 num_train_frames=100000
!modal run modal_off_policy.py 2>&1 | grep -v "modal.com"

✓ Initialized. View run at 
✓ Created objects.
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/modal_off_policy.py
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/cfgs
├── 🔨 Created mount PythonPackage:off_policy, PythonPackage:train_off_policy, 
│   PythonPackage:mw, PythonPackage:utils, PythonPackage:logger, 
│   PythonPackage:replay_buffer, PythonPackage:video
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/demos
└── 🔨 Created function train.
WANDB_API_KEY in env: True
/root/off_policy.py
/root/cfgs/off_policy_config.yaml
/root/off_policy.py
torch version 2.7.1+cu118
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym shou

In [ ]:
# executing for about 1.9 hr
# agent.num_critics=10 utd=5 num_train_frames=40000
!modal run modal_off_policy.py 2>&1 | grep -v "modal.com"

✓ Initialized. View run at 
✓ Created objects.
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/modal_off_policy.py
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/cfgs
├── 🔨 Created mount PythonPackage:off_policy, PythonPackage:train_off_policy, 
│   PythonPackage:mw, PythonPackage:utils, PythonPackage:logger, 
│   PythonPackage:replay_buffer, PythonPackage:video
├── 🔨 Created mount /content/drive/MyDrive/Colab 
│   Notebooks/Stanford_DeepRL_Spring2026/hw2/demos
└── 🔨 Created function train.
WANDB_API_KEY in env: True
/root/off_policy.py
/root/cfgs/off_policy_config.yaml
/root/off_policy.py
torch version 2.7.1+cu118
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym shou

In [ ]:
!mkdir modal_results
!modal volume get cs224r-results / ./modal_results 2>&1 | grep -v "modal.com"

✓ Finished downloading files to local!


Increasing the UTD ratio to 5 further boosts the agent's sample efficiency through accelerated critic updates, and expanding the ensemble to 10 critics strongly suppresses the resulting overestimation via a highly conservative value estimation, thus yielding even better performance.

3. **Comparison and Analysis.**  
Compare the PPO learning curve (`eval/episode_success`) from Problem 2 with the actor-critic curve from Problem 3 (2 critics, UTD=1).  
    <font color="red">**In 3–5 sentences, explain at least two concrete differences you observe between the two algorithms in terms of sample efficiency and final performance. Justify each observed difference by connecting it to a specific property of each algorithm.**</font>  
    The Actor-Critic algorithm from Problem 3 demonstrates vastly superior sample efficiency and final performance compared to the PPO algorithm from Problem 2, with Actor-Critic converging to a 100% success rate within 100k steps, while PPO oscillates around 20% to 40% success throughout 1M steps, only peaking at 50%. This performance gap exists because PPO is an on-policy algorithm that discards collected rollouts immediately after a few gradient updates, forcing it to constantly spend environment steps gathering fresh data. In a continuous control task with sparse rewards like hammering a nail, this data-destructive nature prevents the agent from keeping rare successful trial data in memory, causing it to quickly drift away from the pre-trained expert distribution. On the other hand, the off-policy Actor-Critic leverages a permanent replay buffer to repeatedly sample past and current experiences, alongside a distinct expert demonstration buffer used for periodic behavior cloning updates, ultimately reinforcing successful trajectories. Therefore, the off-policy algorithm achieves significantly higher data efficiency and optimization stability than its on-policy counterpart.